# Continuous LunarLander — PPO

Compare standard engine response with a 0.1 s engine-command lag. Actuator command states are included in the policy observation. PPO is implemented through Stable-Baselines3.

Checkpoint selection maximizes mean return on validation B. Landing rate follows the simulator’s settled-landing definition.

## 1. Environment setup

Mount Google Drive, locate the project, install dependencies, and register the package import path. Local execution uses the project root or its `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

# Project location.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/rl_examples")  # Google Drive project path.
else:
    PROJECT = Path.cwd()
    if not (PROJECT / "pyproject.toml").exists():
        PROJECT = PROJECT.parent

assert (PROJECT / "pyproject.toml").exists(), f"Project not found: {PROJECT}"
%pip -q install "swig>=4.2,<5"
%pip -q install -e "{PROJECT}"

assert (PROJECT / "rl_examples" / "__init__.py").exists(), "Python package directory missing"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import importlib
importlib.invalidate_caches()

## 2. Imports

Load configuration, training, evaluation, simulation, and plotting functions. Limit PyTorch CPU execution to two threads.

In [ ]:
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
from dataclasses import replace
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Video
from rl_examples.configs.settings import NavigationConfig, LanderConfig, SACConfig, PPOConfig, TrainingConfig
from rl_examples.configs.smoke import smoke_settings
from rl_examples.training.runner import run_experiment
from rl_examples.evaluation.comparison import compare_runs
from rl_examples.evaluation.metrics import paired_efficiency
from rl_examples.simulation.examples import simulate_runs
from rl_examples.visualization.plots import plot_learning_curves, plot_trajectories
from rl_examples.utils.labels import run_labels, format_results

torch.set_num_threads(2)

## 3. Experiment configuration

Set environment parameters, algorithm hyperparameters, training budgets, and scenario seeds. `SMOKE_ONLY=True` runs a short pipeline check; `False` enables full training. `MODE="load"` reads completed runs with matching settings. Results are stored under `outputs/`.

In [ ]:
SMOKE_ONLY = True  # True: pipeline check; False: full training.
MODE = "train"     # "train" or "load".
RUN_NAME = "lunarlander_ppo_v1"  # Experiment identifier.
RUN_HELD_OUT_TEST = False

conditions = {
    "standard": LanderConfig(tau_seconds=0.0),
    "command_lag": LanderConfig(tau_seconds=0.10),
}
stages = None
agent_config = PPOConfig(n_steps=1000, batch_size=250)

training_config = TrainingConfig(
    steps_per_stage=250_000, eval_every=10_000, seed=42,
    early_stopping=True, min_steps=100_000, patience=10, min_delta=2.0,
    validation_a=tuple(range(20000, 20020)),
    validation_b=tuple(range(21000, 21020)),
    test_seeds=tuple(range(30000, 30100)),
)

if SMOKE_ONLY:
    conditions, agent_config, training_config, stages = smoke_settings(
        conditions, agent_config, training_config, stages)
OUTPUT = PROJECT / "outputs" / (RUN_NAME + ("_smoke" if SMOKE_ONLY else ""))
print(OUTPUT)

## 4. Training

Train an independent policy for each condition and evaluate after each training interval. Validation A tracks performance; validation B selects the checkpoint. Training stops early when validation B return has not improved by at least `min_delta` for `patience` consecutive evaluations after `min_steps`.

In [ ]:
runs = run_experiment(conditions, agent_config, training_config, OUTPUT, stages=stages, mode=MODE)

## 5. Learning curves

Plot exploratory training returns and deterministic validation returns and landing rates.

In [ ]:
labels = run_labels(runs)
fig = plot_learning_curves(runs)
fig.savefig(OUTPUT / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Checkpoint validation

Compare selected and final checkpoints on validation B. Report efficiency on scenarios completed successfully by both selected policies. These selection-panel scores are not held-out test estimates.

In [ ]:
summary, episodes = compare_runs(runs, training_config.validation_b)
display(format_results(summary, labels))
paired = paired_efficiency(episodes[episodes.checkpoint == "best"])
display(format_results(paired, labels))
summary.to_csv(OUTPUT / "validation_summary.csv", index=False)

## 7. Trajectory comparison

Simulate the selected policies on three fixed validation scenarios. Rows correspond to scenario seeds; columns correspond to conditions. Each panel reports the episode outcome.

In [ ]:
SCENARIO_SEEDS = (21000, 21001, 21002)
rollouts = simulate_runs(runs, seeds=SCENARIO_SEEDS)
fig = plot_trajectories(rollouts)
fig.savefig(OUTPUT / "trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Video export

Export selected-policy rollouts as MP4 files with PNG previews and JSON run metadata. Enable after full training. Navigation is sampled once per high-level action (2 fps); LunarLander is recorded at 50 fps.

In [ ]:
RECORD_VIDEOS = False
VIDEO_SEEDS = (21000, 21001)
if RECORD_VIDEOS:
    if SMOKE_ONLY:
        raise ValueError("Video demonstrations require SMOKE_ONLY=False and completed training")
    videos = simulate_runs(runs, seeds=VIDEO_SEEDS, video_dir=OUTPUT / "videos")
    for condition, scenarios in videos.items():
        for seed, result in scenarios.items():
            print(f"{labels[condition]} | seed = {seed} | {result['metrics']['outcome']}")
            display(Video(str(result["video_path"]), embed=True))

## 9. Held-out evaluation

Evaluate selected checkpoints on reserved test scenarios after fixing the experiment settings. Report outcomes and paired efficiency without further checkpoint selection.

In [ ]:
if RUN_HELD_OUT_TEST:
    if SMOKE_ONLY:
        raise ValueError("Held-out evaluation requires full training")
    test_summary, test_episodes = compare_runs(
        runs, training_config.test_seeds, split="test", checkpoints=("best",))
    display(format_results(test_summary, labels))
    display(format_results(paired_efficiency(test_episodes), labels))
    test_summary.to_csv(OUTPUT / "test_summary.csv", index=False)
else:
    print("Held-out evaluation disabled.")